In [ ]:
#TODO
# SAM, MedSAM(fine-tuned decoder), SAMMed2D(fine-tuned all) results over 51 datasets

#### How can we fine tune SAM efficitively with the least data?  

1. Will different selection query give us diferent results?    
constraint:  
 dataset size:51 (40 train; 11 test)  
 epochs: 1000 for add one sample 
 fixed random seed  
 model: SAM  
 checkpoint: b  
 fine-tuning structure: mask decoder  

* baseline: random selection from 40 dataset
* entropy with dropout 
* entropy with dropout + artifacts
* encoder of SAM + clustering  



- Plot: Accuracy on the 11 test set as a function of number of selected samples (training)
- ID of selected samples for each iteration

In [1]:
import datasets.path as path
# import nibabel as nib
from glob import glob
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import monai
# from utils.SurfaceDice import compute_dice_coefficient
from tqdm import tqdm
import json
from segment_anything import SamPredictor, sam_model_registry
from segment_anything.utils.transforms import ResizeLongestSide
import matplotlib.pyplot as plt
import wandb
from utils import NpzDataset, get_bbox_from_mask
from metrics import compute_dice_coefficient
from strategy import random_sampling
from eval import infer, log_image_table

In [2]:
# training and sampling dataset path prefix
prefix = './datasets/RAINE_organ_51'
task = 'MRI_Pancreas'
training_pool_path = os.path.join(prefix, task, 'train')
# label id
# left kidney: 1,
# right kidney: 2,
# pancreas: 3,
# background: 0,
label_id = 3

#SAM MODEL TYPE 
sam_model_type = 'vit_b'
# SAM checkpoint
checkpoint = './checkpoints/SAM/sam_vit_b_01ec64.pth'
# device 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# sample strategy
strategy = 'random'
# active learning fine tunning checkpoint
base_model = 'SAM'
save_path_ckp = os.path.join('./checkpoints/', base_model + '_' + task + '_' + strategy)
os.makedirs(save_path_ckp, exist_ok=True)
# sampling dataset path
sampling_datapath = os.path.join(prefix, task, base_model+'_sampling')
os.makedirs(sampling_datapath, exist_ok=True)

In [3]:
training_pool = glob(os.path.join(training_pool_path, '*.npz'))
sample_pool = []
num_epochs = 150
batch_size = 16 # make sure batch size is smaller than sample slices
losses = []
best_loss = 1e10
sam_model = sam_model_registry[sam_model_type](checkpoint=checkpoint).to(device)

# Set up the optimizer, hyperparameter tuning will improve performance here
optimizer = torch.optim.Adam(sam_model.mask_decoder.parameters(), lr=1e-5, weight_decay=0)
seg_loss = monai.losses.DiceCELoss(sigmoid=True, squared_pred=True, reduction='mean')

save_path_ckp_epoch = os.path.join(save_path_ckp, f'epochs{num_epochs}')
os.makedirs(save_path_ckp_epoch, exist_ok=True)

# wandb
# wandb.init(
#     # Set the project where this run will be logged
#     project="SAM-Activelearning", 
#     group="random",
#     name=f'{base_model}_{sam_model_type}_{task}_{strategy}_epoch{num_epochs}',
#     config = {
#         "task": task,
#         'label_id': label_id,
#         'base_model': base_model,
#         "model": sam_model_type,
#         "strategy": strategy,
#         "num_epochs": num_epochs
#     }
# )


In [ ]:
sam_model.train()
for i in range(len(training_pool)):
    sample_pool, training_pool = random_sampling(training_pool, sample_pool)
    print('num samples:', len(sample_pool))
    print('num training:', len(training_pool))
    num_samples = len(sample_pool)
    if num_samples > 2 and num_samples <= 5:
        batch_size = 32
    elif num_samples > 5:
        batch_size = 64
    
    # print('Number of samples: ', num_samples, '\tBatch size: ', batch_size)
    sample_dataset = NpzDataset(sample_pool)
    sample_dataloader = DataLoader(sample_dataset, batch_size=batch_size, shuffle=True)
    for epoch in range(num_epochs):
        epoch_loss = 0
        for step, (image_embedding, gt2D, boxes) in enumerate(tqdm(sample_dataloader)):
            # img_embed: (B, 256, 64, 64), gt2D: (B, 1, 256, 256), bboxes: (B, 4)
            # print(f"{image_embedding.shape=}, {gt2D.shape=}, {boxes.shape=}")
            with torch.no_grad():
                box_np = boxes.numpy() # [0, 0, 256, 256]
                sam_trans = ResizeLongestSide(sam_model.image_encoder.img_size)
                box = sam_trans.apply_boxes(box_np, (gt2D.shape[-2], gt2D.shape[-1]))
                box_torch = torch.as_tensor(box, dtype=torch.float, device=device)
                if len(box_torch.shape) == 2:
                    box_torch = box_torch[:, None, :] # (B, 1, 4)
                # get prompt embeddings 
                sparse_embeddings, dense_embeddings = sam_model.prompt_encoder(
                    points=None,
                    boxes=box_torch,
                    masks=None,
                )
            # predicted masks
            # print(f"{image_embedding.shape=}, {sparse_embeddings.shape=}, {dense_embeddings.shape=}")
            mask_predictions, _ = sam_model.mask_decoder(
                image_embeddings=image_embedding.to(device), # (B, 256, 64, 64)
                image_pe=sam_model.prompt_encoder.get_dense_pe(), # (1, 256, 64, 64)
                sparse_prompt_embeddings=sparse_embeddings, # (B, 2, 256)
                dense_prompt_embeddings=dense_embeddings, # (B, 256, 64, 64)
                multimask_output=False,
            )

            loss = seg_loss(mask_predictions, gt2D.to(device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        epoch_loss /= step
        losses.append(epoch_loss)
        print(f'EPOCH: {epoch}, Loss: {epoch_loss}')
        wandb.log({f"Train/loss of {num_samples} num sample": epoch_loss})
        # save the latest model checkpoint
        torch.save(sam_model.state_dict(), os.path.join(save_path_ckp_epoch, f'sam_{sam_model_type}_{num_samples:02d}_latest.pth'))
        # save the best model
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save(sam_model.state_dict(), os.path.join(save_path_ckp_epoch, f'sam_{sam_model_type}_{num_samples:02d}_best.pth'))

    # plot loss
    # plt.plot(losses)
    # plt.title('Dice + Cross Entropy Loss')
    # plt.xlabel('Epoch')
    # plt.ylabel('Loss')
    # # plt.show() # comment this line if you are running on a server
    # plt.savefig(os.path.join(save_path_ckp, f'sam_{sam_model_type}_{num_samples:02d}_train_loss.png'))
    # plt.close()

with open(os.path.join(sampling_datapath, f'{strategy}_pool.json'), 'w') as f:
    json.dump(sample_pool, f, indent=4)
    
# wandb.finish()

In [4]:
!export WANDB_BASE_URL=http://172.17.0.1:8080
!export WANDB_API_KEY=4aaa2e71cdec13a78a42c6ceac38dd0c7235a131

wandb.login()

2023-12-04 16:17:25,490 - Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


False

In [6]:
# eval training pool
training_pool = glob(os.path.join(training_pool_path, '*.npz'))
dice_train = []

num_epochs = 100
ckp_paths = sorted(glob(os.path.join(save_path_ckp, f'epochs{num_epochs}', '*latest.pth')))
for p in tqdm(ckp_paths):
    avg_dice = []
    for t in tqdm(training_pool):
        id = os.path.basename(t).split('.')[0]
        imgs = np.load(t)['imgs']

        dataset = NpzDataset([t])
        batch_size = len(dataset)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False) # make sure batch size is larger than sample slices
        for embeddings, gts, _ in dataloader:
            preds, _ = infer(imgs, embeddings, p, sam_model_type, device)
        gts = gts[:, 0, :, :].numpy() # convert tensor to numpy (B, H, W)
        dice = compute_dice_coefficient(gts, preds)
        avg_dice.append(dice)
    dice_train.append(np.mean(avg_dice))

plt.plot(dice_train)
plt.xlabel('Number of samples')
plt.ylabel('Dice coefficient')
plt.title(f'{base_model} {task} {strategy} sampling')
plt.show()
plt.savefig(os.path.join('figures/train', f'{base_model}_{task}_{strategy}_epochs{num_epochs}_sampling.png'))
plt.close()

  0%|          | 0/40 [00:00<?, ?it/s]

In [ ]:
# eval on test set

num_epochs = 150
ckp_paths = sorted(glob(os.path.join(save_path_ckp, f'epochs{num_epochs}', '*latest.pth')))
testing_pool_path = os.path.join(prefix, task, 'test')
testing_pool = glob(os.path.join(testing_pool_path, '*.npz'))
dice_test = []

# wandb
wandb.init(
    # Set the project where this run will be logged
    project="SAM-Activelearning", 
    group="random",
    name=f'{base_model}_{sam_model_type}_{task}_{strategy}_epoch{num_epochs}',
    config = {
        "task": task,
        'label_id': label_id,
        'base_model': base_model,
        "model": sam_model_type,
        "strategy": strategy,
        "num_epochs": num_epochs
    }
)
table = wandb.Table(columns=["image", "pred", "gt", 'id', 'dice'])

# eval on test set for each checkpoint
for p in tqdm(ckp_paths):
    avg_dice = []
    for t in tqdm(testing_pool):
        id = os.path.basename(t).split('.')[0]
        imgs = np.load(t)['imgs']

        dataset = NpzDataset([t])
        batch_size = len(dataset)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False) # make sure batch size is larger than sample slices
        for embeddings, gts, _ in dataloader:
            preds, _ = infer(imgs, embeddings, p, sam_model_type, device)
        gts = gts[:, 0, :, :].numpy() # convert tensor to numpy (B, H, W)
        dice = compute_dice_coefficient(gts, preds)
        avg_dice.append(dice)
        if '40' in p:
            preds_int = preds * 1 # convert boolean to int
            log_image_table(imgs, preds_int, gts, id, dice, table)
    dice_test.append(np.mean(avg_dice))
    wandb.log({f"Eval/Dice {strategy} epoch{num_epochs}": np.mean(avg_dice)}, commit=True)
wandb.log({f"predictions_table {strategy} epoch{num_epochs}":table}, commit=True)
wandb.finish()
    

In [ ]:
plt.plot(dice_test)
plt.xlabel('Number of samples')
plt.ylabel('Dice coefficient')
plt.title(f'{base_model} {task} {strategy} sampling')
plt.show()
plt.savefig(os.path.join('figures/eval', f'{base_model}_{task}_{strategy}_epochs{num_epochs}_sampling.png'))
plt.close()